# R14 · Lymph node 5k: top-400 variable vs top-400 expressed genes (supplementary Fig S12)

Fig S1's readouts for lymph node 5k under two gene choices of 400 genes: the index's (top variance, `topvar_genes`)
and the 400 genes with the highest mean count. Each is the production build with only the gene list changed
(`gene_choice.build`: `build_quadtree_tiles`, `build_tile_covs_full`, seed-0 split, `build_layout`); the
top-variance rebuild is checked against the cached layout. Readouts as R10 (`block_structure`): every index tile's
mean |r| within and between blocks (pairs with a gene without counts in the tile left out), and the largest niche's
mean correlation matrix in its gene order with its blocks.

* **Prerequisites:** `R0_index_cache` (lymph node 5k seed-0 layout, for the check).
* **Writes:** `results/ln5k_gene_choice/tile_means.csv` (per scenario and index tile), `niche_example_{var,mean}.npz`,
  `genes.csv` (per scenario: gene, mean, variance).
* **Runtime / memory:** ~10 min; ~30 GB (one production build at a time).
  `sbatch --cpus-per-task=8 --mem=64G --job-name=R14_ln5k_gene_choice --output=results/_logs/R14_ln5k_gene_choice/%x_%j.out experiments/common/run_nb.sbatch experiments/compute/R14_ln5k_gene_choice.ipynb`

In [1]:
import os
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets.yaml").exists())
sys.path.insert(0, str(ROOT / "experiments" / "common"))
import numpy as np
import pandas as pd

import block_structure as bs
import data
import gene_choice as gc
import results_io as rio

TOPIC = "ln5k_gene_choice"
KEY = "lymph_node_5k"
N_GENES = 400
SCENARIOS = {"var": "top-400 variable", "mean": "top-400 expressed"}
FORCE = False
N_JOBS = int(os.environ.get("SLURM_CPUS_PER_TASK", 4))
OUT = rio.topic(TOPIC)
print(data.stamp(), f"{N_JOBS} cores")

{'tag': 'method-v2', 'describe': 'method-v3-4-gb026ef3-dirty', 'commit': 'b026ef3'} 8 cores


## Builds and readouts

In [2]:
if FORCE or not (OUT / "tile_means.csv").exists():
    tag = data.stamp()["tag"]
    means, genes = [], []
    for sc in SCENARIOS:
        b = gc.build(KEY, N_GENES, sc, n_jobs=N_JOBS)
        L = b["layout"]
        if sc == "var":  # the index's own gene choice: must be the cached build
            c = data.load_cache(KEY, data.EXAMPLE_SEED, parts=("layout",))
            assert (b["genes"] == c.genes).all() and (b["train"] == c.train).all()
            assert np.array_equal(L.labels, c.layout.labels)
            assert all(np.array_equal(L.perm_list[k], c.layout.perm_list[k])
                       and list(map(tuple, L.block_dict[k])) == list(map(tuple, c.layout.block_dict[k]))
                       for k in c.layout.block_dict)
            print("top-variance rebuild == cached layout (niches, gene orders, blocks)")
        tm = bs.tile_means(L, b["train_covs"], KEY).assign(scenario=sc, tile=lambda d: b["train"][d.row])
        means.append(tm)
        ex = bs.niche_example(L, b["train_covs"])
        ex["genes"] = b["genes"][np.asarray(L.perm_list[ex["niche"]])]
        np.savez_compressed(OUT / f"niche_example_{sc}.npz", **ex)
        rio.register(TOPIC, OUT / f"niche_example_{sc}.npz",
                     f"{SCENARIOS[sc]} genes: the largest niche's mean correlation matrix (float32) in its gene order, "
                     "block runs, niche, tiles, gene names (as R10's niche_example)")
        genes.append(pd.DataFrame({"scenario": sc, "rank": np.arange(len(b["genes"])), "gene": b["genes"],
                                   "mean": b["mean"], "var": b["var"]}))
        print(f"{sc}: {len(set(L.labels.tolist()))} niches, {sum(len(r) for r in L.block_dict.values())} blocks", flush=True)
        del b, L
    rio.write_csv(pd.concat(means, ignore_index=True).assign(tag=tag), TOPIC, "tile_means.csv",
                  "per scenario (var: top-400 variable genes, the index's; mean: top-400 expressed) and index tile "
                  "(seed-0 split): niche, mean |r| over its within-block and between-block gene pairs (pairs with a gene "
                  "without counts in the tile left out), pair counts, genes with counts")
    rio.write_csv(pd.concat(genes, ignore_index=True).assign(tag=tag), TOPIC, "genes.csv",
                  "per scenario: the 400 genes in selection order with their mean and variance over all cells")
tm = pd.read_csv(OUT / "tile_means.csv")
gs = pd.read_csv(OUT / "genes.csv")

[2026-11-11 23:20:02,861] INFO spindle_dev.preprocessing: build_quadtree_tiles: dropping 366 low-density tile(s) (19750 spots)
/home/NAShome/ghoss18/miniforge3/envs/spindle_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/NAShome/ghoss18/miniforge3/envs/spindle_env/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
[2026-11-11 23:21:33,959] INFO spindle_dev.layout: Leiden resolution 1.30: 15 niches, sizes [873, 861, 777, 593, 527, 504, 443, 421, 366, 321, 314, 303, 274, 199, 196]
[2026-11-11 23:21:46,464] INFO spindle_dev.layout: Niche 0: 20 blocks
[2026-11-11 23:21:46,567] INFO spindle_dev.layout: Niche 1: 22 blocks
[2026-11-11 23:21:46,649] INFO spindle_dev.layout: Niche 2: 20 blocks
[2026-

top-variance rebuild == cached layout (niches, gene orders, blocks)
var: 15 niches, 302 blocks


[2026-11-11 23:22:20,494] INFO spindle_dev.preprocessing: build_quadtree_tiles: dropping 366 low-density tile(s) (19750 spots)
/home/NAShome/ghoss18/miniforge3/envs/spindle_env/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
[2026-11-11 23:23:22,919] INFO spindle_dev.layout: Leiden resolution 0.80: 12 niches, sizes [980, 961, 877, 756, 644, 498, 491, 472, 461, 352, 260, 220]
[2026-11-11 23:23:35,409] INFO spindle_dev.layout: Niche 0: 23 blocks
[2026-11-11 23:23:35,494] INFO spindle_dev.layout: Niche 1: 21 blocks
[2026-11-11 23:23:35,574] INFO spindle_dev.layout: Niche 2: 21 blocks
[2026-11-11 23:23:35,660] INFO spindle_dev.layout: Niche 3: 20 blocks
[2026-11-11 23:23:35,766] INFO spindle_dev.layout: Niche 4: 23 blocks
[2026-11-11 23:23:35,847] INFO spindle_dev.layout: Niche 5: 21 blocks
[2026-11-11 23:23:35,928] INFO spindle_dev.layout: Niche 6: 20 blocks
[2026-11-11 23:23:36,030]

mean: 12 niches, 259 blocks
wrote results/ln5k_gene_choice/tile_means.csv: 13944 rows
wrote results/ln5k_gene_choice/genes.csv: 800 rows


## Summary (shown, not saved)

In [3]:
g = tm.groupby("scenario", sort=False)
shared = len(set(gs.gene[gs.scenario == "var"]) & set(gs.gene[gs.scenario == "mean"]))
print(f"genes in both scenarios: {shared} of {N_GENES}")
pd.DataFrame({"tiles": g.size(), "niches": g.niche.nunique(),
              "within": g.mean_within.mean(), "within_sd": g.mean_within.std(),
              "between": g.mean_between.mean(), "between_sd": g.mean_between.std(),
              "ratio": g.mean_within.mean() / g.mean_between.mean(),
              "within > between (%)": 100 * g.apply(lambda d: (d.mean_within > d.mean_between).mean()),
              "genes with counts": g.genes_expressed.median()}).round(3)

genes in both scenarios: 365 of 400


,tiles,niches,within,within_sd,between,between_sd,ratio,within > between (%),genes with counts
scenario,,,,,,,,,
var,6972,15,0.122,0.025,0.085,0.011,1.438,100.000,396.0
mean,6972,12,0.113,0.019,0.086,0.011,1.310,99.986,400.0


In [4]:
rio.write_manifest(TOPIC, "R14_ln5k_gene_choice.ipynb",
                   inputs=["the lymph node 5k section (datasets.yaml)", "results/_cache/lymph_node_5k/seed0/layout.pkl (R0, check)"])

wrote results/ln5k_gene_choice/manifest.json: 4 files; peak memory 27.21 GB


{'topic': 'ln5k_gene_choice',
 'notebook': 'R14_ln5k_gene_choice.ipynb',
 'date': '2026-11-11 23:24',
 'tag': 'method-v2',
 'describe': 'method-v3-4-gb026ef3-dirty',
 'commit': 'b026ef3',
 'inputs': ['the lymph node 5k section (datasets.yaml)',
  'results/_cache/lymph_node_5k/seed0/layout.pkl (R0, check)'],
 'peak_rss_gb': 27.21,
 'notes': '',
 'files': {'genes.csv': {'description': 'per scenario: the 400 genes in selection order with their mean and variance over all cells',
   'rows': 800,
   'bytes': 36492},
  'niche_example_mean.npz': {'description': "top-400 expressed genes: the largest niche's mean correlation matrix (float32) in its gene order, block runs, niche, tiles, gene names (as R10's niche_example)",
   'rows': None,
   'bytes': 566778},
  'niche_example_var.npz': {'description': "top-400 variable genes: the largest niche's mean correlation matrix (float32) in its gene order, block runs, niche, tiles, gene names (as R10's niche_example)",
   'rows': None,
   'bytes': 57061